### Ler etapa bronze

In [0]:
df_orders = spark.table(
    "workspace.bronze.orders"
)

### Analisar dados

In [0]:
display(df_orders)

In [0]:
df_orders.printSchema()

In [0]:
print(
    f"Quantidade de registros: {df_orders.count()}"
)

### Procurar nulos

In [0]:
from pyspark.sql.functions import *

In [0]:
display(
    df_orders.select(
        [
            count(
                when(
                    col(c).isNull(),
                    c
                )
            ).alias(c)
            for c in df_orders.columns
        ]
    )
)

### Procurar duplicidades

In [0]:
total = df_orders.count()

print(
    f"Total registros: {total}"
)

In [0]:
distintos = df_orders.distinct().count()

print(
    f"Registros distintos: {distintos}"
)

### Remover duplicidades

In [0]:
df_orders_silver = (
    df_orders
    .dropDuplicates()
)

### Criar metadado na Silver

In [0]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, lit

In [0]:
df_orders_silver = (
    df_orders_silver
    .withColumn(
        "data_processamento_silver",
       from_utc_timestamp(
           current_timestamp(),
           "America/Sao_Paulo"
       )
    )
    .withColumn(
        "camada",
        lit("silver")
    )
)

### Verificar resultado

In [0]:
display(df_orders_silver)

### Gravar na Silver

In [0]:
(
    df_orders_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.orders"
    )
)

### Validação


In [0]:
%sql
select
*
from
workspace.silver.orders